In [135]:
import os 
import sqlalchemy as sa 
import pandas as pd 
from sqlalchemy import create_engine, text

import yaml

In [33]:
with open('../configs/dataset_creation.yaml') as file:
    config = yaml.full_load(file)

In [209]:
def get_conn_url_from_path(file_path):
    """Returns a connection URL for filedatabases 
    based on the database file path. (To be used within a SQLAlchemy Engine.)
    ---
    Included databases: MSAccess (*.mdb, *.accdb) + SQLite
    """
    ext = os.path.splitext(file_path)[1]

    if ext in ['.mdb', '.accdb']:
        connection_string = (
            rf"DRIVER={{Microsoft Access Driver (*.mdb, *.accdb)}};"
            rf"DBQ={file_path};"
            rf"ExtendedAnsiSQL=1;" # to support DECIMAL(m, n) columns in Access
        )

        # Create URL without usr and pwd 
        connection_url = sa.engine.URL.create(
        "access+pyodbc",
        query={"odbc_connect": connection_string}
        )
    elif ext == '.sqlite':
        connection_url = rf"sqlite:///{path}"

    return connection_url

In [130]:
path = os.path.join(config['data_dir'], config['datasets']['bwi_2002'])
engine_bwi_2002 = sa.create_engine(get_conn_url_from_path(path))

In [81]:
insp = sa.inspect(engine_bwi_2002)

In [82]:
insp.get_table_names()

['b0_ecke',
 'b0_tab',
 'b2_ba_natwg',
 'b2_ba_natwg_2012',
 'b2_baeume',
 'b2_Baeume_korr',
 'b2_baeume_var',
 'b2_bestand',
 'b2_bestock',
 'b2_bestock_2012',
 'b2_bestock_baanteile',
 'b2_bestock_gt4m',
 'b2_bestock_gt4m_2012',
 'b2_bestock_gt4m_ba',
 'b2_bestock_le4m',
 'b2_bestock_le4m_2012',
 'b2_bestock_le4m_ba',
 'b2_boden',
 'b2_boden_quer',
 'b2_def_banwg',
 'b2_ecke_raum',
 'b2_ecke_w',
 'b2_idbestand',
 'b2_jung',
 'b2_tab',
 'b2_tab_w',
 'b2_tot_ge20cm',
 'b2_weg',
 'b2_weg_eg',
 'b2_wfkt_ls',
 'b2_wfkt_quer_ls',
 'b2_wrand',
 'b2_wrand_ab',
 'b2_wzp']

In [172]:
with open('../configs/dataset_creation.yaml') as file:
    config = yaml.full_load(file)

In [162]:
query = config['queries']['bwi_2002_ecke']
formatted_query = query.replace('\n', ' ').replace('  ', ' ')
df_ecke = pd.read_sql(formatted_query, engine_bwi_2002)

In [163]:
query = config['queries']['bwi_2002_tab']
formatted_query = query.replace('\n', ' ').replace('  ', ' ')
df_tab = pd.read_sql(formatted_query, engine_bwi_2002)

In [164]:
query = config['queries']['bwi_2002_baeume']
formatted_query = query.replace('\n', ' ').replace('  ', ' ')
df_baeume = pd.read_sql(formatted_query, engine_bwi_2002)

In [142]:
df_joined = pd.merge(df_ecke, df_tab, on='Tnr', how='inner')

In [168]:
df_joined_x = pd.merge(df_baeume,  df_joined, on=['Tnr', 'Enr'], how='inner')

In [173]:
query = config['queries']['bwi_2002_bestand']
formatted_query = query.replace('\n', ' ').replace('  ', ' ')
df_bestand = pd.read_sql(formatted_query, engine_bwi_2002)

In [174]:
df_joined_x = pd.merge(df_joined_x,  df_bestand, on=['Tnr', 'Enr'], how='inner')

In [182]:
columns = insp.get_columns('b2_jung')

for column in columns:
    print(f"  {column['name']} ({column['type']})")

  Tnr (INTEGER)
  Enr (SMALLINT)
  Bnr (SMALLINT)
  Bs (SMALLINT)
  Ba (SMALLINT)
  Gr (SMALLINT)
  Verbiss (SMALLINT)
  Schael (SMALLINT)
  Sonst (SMALLINT)
  Schu (SMALLINT)
  Anz (SMALLINT)


In [ ]:
query = config['queries']['bwi_2002_bestand']
formatted_query = query.replace('\n', ' ').replace('  ', ' ')
df_bestand = pd.read_sql(formatted_query, engine_bwi_2002)

Column: Tnr, Type: INTEGER
Column: Enr, Type: SMALLINT
Column: VBl, Type: SMALLINT
Column: InvE1, Type: SMALLINT
Column: InvE2, Type: SMALLINT
Column: InvE2x, Type: SMALLINT
Column: InvE2y, Type: SMALLINT
Column: InvE2z, Type: SMALLINT
Column: InvE3, Type: SMALLINT
Column: InvE3w, Type: SMALLINT
Column: Use12, Type: SMALLINT
Column: Use12z, Type: SMALLINT
Column: Use22y, Type: SMALLINT
Column: Use22z, Type: SMALLINT
Column: Use13, Type: SMALLINT
Column: Use23, Type: SMALLINT
Column: Use23w, Type: SMALLINT
Column: Use2y3, Type: SMALLINT
Column: Use2z3, Type: SMALLINT
Column: Use123, Type: SMALLINT
Column: Use22z3, Type: SMALLINT
Column: Bl, Type: SMALLINT
Column: Gform, Type: SMALLINT
Column: Gneig, Type: SMALLINT
Column: Gexp, Type: SMALLINT
Column: natHoe, Type: SMALLINT
Column: GneigKl5, Type: SMALLINT
Column: GExpKl4, Type: SMALLINT
Column: GExpKl8, Type: SMALLINT
Column: potNatWg, Type: SMALLINT


In [214]:
path = os.path.join(config['data_dir'], config['datasets']['climate'])
engine_climate = sa.create_engine(get_conn_url_from_path(path))

In [215]:
insp_climate = sa.inspect(engine_climate)

In [216]:
table_names = insp_climate.get_table_names()

In [220]:
insp_climate.has_table('climate_monthly')

False